<center><h1>Анализ датасета</h1><center>

In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv("hirehi_parsing.csv")
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,https://hirehi.ru/analytics/sistemnyi-analitik...


In [3]:
df.isna().sum()

id           0
title        0
category     0
company      0
format       0
level        0
salary      46
link         0
dtype: int64

<center><strong>Заметим, что пропуски у нас есть только в столбце "salary"</br>
Попозже разберемся как их заполнить</strong></center>


In [4]:
df.duplicated().sum()

np.int64(0)

<center><strong>дубликатов нет, хорошо</strong></center>

In [5]:
df["category"].value_counts()

category
development    3612
analytics      1325
management     1116
qa              914
design          753
devops          596
marketing       438
Name: count, dtype: int64

Видим 7 категорий в "category"

<h3><strong>Займемся обработкой "salary", знаем, что есть 46 пропусков, на первый взгляд видим, что есть данные в таких форматах 'от 350 000 ₽', '~ 223 000 ₽', 'от 220 000 до 268 000 ₽', 'от 180 до 220 ₽'. Для начала оставим только цифры, а также посмотрим, есть ли в наших данных зарплаты в долларах.</strong></h3> 

In [6]:
pd.Series(df["salary"].unique())

0                  от 240 000 ₽
1                  от 350 000 ₽
2                   ~ 372 000 ₽
3                   ~ 223 000 ₽
4                   ~ 210 000 ₽
                 ...           
3399    от 240 000 до 320 000 ₽
3400     от 80 000 до 160 000 ₽
3401            от 180 до 220 ₽
3402    от 200 000 до 285 000 ₽
3403    от 220 000 до 268 000 ₽
Length: 3404, dtype: str

In [7]:
dollarorruble = df["salary"].str.contains(r"\$|₽").sum()
dollarorruble

np.int64(8488)

Знаем, что у нас 8754 строки, 8488 строк с рублями/долларами, тогда что в оставшихся 266 строках? 

In [8]:
not_dollarorruble = df["salary"].str.contains(r"\$|₽") == False

In [9]:
df[not_dollarorruble]["salary"].unique()

<StringArray>
['зп не указана', nan, 'зп не указана ', 'не оплачивается']
Length: 4, dtype: str

Видим, что оставшиеся ячейки это либо пропуски, либо зарплата не указана явно, можно приравнять эти строки к пропускам, чтобы потом заполнить их по общему правилу. 

In [10]:
df["salary"] = df["salary"].replace(['зп не указана','зп не указана ', 'не оплачивается'], np.nan)
df["salary"].isna().sum()

np.int64(266)

теперь получили 266 пропусков в столбце salary, вернемся к ним позже, пока займемся приведением зарплат к числам. </br> Нужно как-то склеить тысячи, чтобы при последующей замене на пробелы всего, кроме цифр, вилки не склеились в одно целое. 

In [11]:
df['salary'] = df['salary'].str.replace(r"[^\d]", " ", regex=True).str.strip()


In [12]:
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240 000,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350 000,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372 000,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223 000,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210 000,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,180 220,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150 000,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,200 000 285 000,https://hirehi.ru/analytics/sistemnyi-analitik...


там, где мы заменили буквы, осталось несколько пробелов, пожэтому заменим их на один, а единственный между тысячами("000") в вилках уберем. Но чтобы нужные пробелы не удалились, заменим их на временные разделители, которые потом тоже удалим.</br>Чтобы понять, как обнаруживать два и более пробелов подряд, использовал эту <a href = "https://habr.com/ru/articles/349860/">статью</a>

In [13]:
df["salary"] = df["salary"].str.replace(r"\s{2,}", "*", regex=True)
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240 000,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350 000,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372 000,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223 000,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210 000,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,180*220,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150 000,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,200 000*285 000,https://hirehi.ru/analytics/sistemnyi-analitik...


заменим двойные и больше пробелы на "*", чтобы не склеилось все, дальше удалим одиночные пробелы. 

In [14]:
df["salary"] = df["salary"].str.replace(r"\s", "", regex=True)
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240000,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350000,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372000,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223000,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210000,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,180*220,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150000,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,200000*285000,https://hirehi.ru/analytics/sistemnyi-analitik...


In [15]:
df["salary"] = df["salary"].str.replace("*", " ")
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240000,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350000,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372000,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223000,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210000,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,180 220,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150000,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,200000 285000,https://hirehi.ru/analytics/sistemnyi-analitik...


In [16]:
pd.Series(df["salary"])

0              240000
1              350000
2              372000
3              223000
4              210000
            ...      
8749          180 220
8750           150000
8751              NaN
8752    200000 285000
8753    220000 268000
Name: salary, Length: 8754, dtype: str

In [17]:
df["salary"] = df["salary"].str.strip()
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240000,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350000,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372000,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223000,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210000,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,180 220,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150000,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,200000 285000,https://hirehi.ru/analytics/sistemnyi-analitik...


Видим, что остались вилки, вместо них подставим среднее значение. Также умножим на 1000 те результаты, которые окажутся меньше 1000

In [18]:
def count_avg(i):
    fork = str(i).split()
    values = []
    for j in fork:
        values.append(float(j))
    if len(values) == 2:
        salary = (values[0] + values[1])/2
    elif len(values) == 1:
        salary = values[0]
    else:
        return np.nan
    
    if salary <=1000:
        salary = salary * 1000

    return salary



In [19]:
df["salary"] = df["salary"].apply(count_avg)
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240000.0,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350000.0,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372000.0,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223000.0,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210000.0,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,200000.0,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150000.0,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,NaN,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,242500.0,https://hirehi.ru/analytics/sistemnyi-analitik...


<h2><strong>Теперь заполним пропуски, для этого будем смотреть category и level и будем рассчитывать число для пропуска исходя из зарплат в этой категории и этого уровня.</strong></h2>

In [20]:
df["level"].value_counts()

level
middle        4235
senior        3089
lead           768
junior         468
intern         129
head            63
грейда нет       1
не указан        1
Name: count, dtype: int64

Так мы нашли что есть несколько грейдов:</br>
1) intern</br>
2) junior</br>
3) middle</br>
5) senior</br> 
5) lead</br>
5) head</br>

In [21]:
head = df["level"].str.contains("head")
df_head = df[head]
df_head

,id,title,category,company,format,level,salary,link
720,35969,Tech lead,development,Аддамант,удалённо,head,360000.0,https://hirehi.ru/development/tech-lead-35969
760,35926,head of marketing,marketing,NDA,удалённо,head,200000.0,https://hirehi.ru/marketing/head-of-marketing-...
979,35694,product designer,design,1inch,офис Дубай,head,595000.0,https://hirehi.ru/design/product-designer-35694
1368,35280,директор по маркетингу,marketing,NDA,офис,head,376000.0,https://hirehi.ru/marketing/direktor-po-market...
1463,35180,cpo,management,Плати по миру,офис Москва,head,500000.0,https://hirehi.ru/management/cpo-35180
...,...,...,...,...,...,...,...,...
7836,26835,cpo,management,Сбер,офис Москва,head,584300.0,https://hirehi.ru/management/cpo-26835
8037,26544,cto,development,NDA,удалённо,head,791400.0,https://hirehi.ru/development/cto-26544
8109,26425,product analyst,analytics,Sweed,удалённо,head,494600.0,https://hirehi.ru/analytics/product-analyst-26425
8674,25577,backend разработчик,development,Т-Банк,гибрид Москва,head,715300.0,https://hirehi.ru/development/backend-razrabot...


In [22]:
nograde = df["level"].str.contains("грейда нет")
df_nograde = df[nograde]
df_nograde

,id,title,category,company,format,level,salary,link
2147,34417,AI Frontend Design Engineer,design,EGO digital,удалённо,грейда нет,240000.0,https://hirehi.ru/design/ai-frontend-design-en...


In [23]:
none = df["level"].str.contains("не указан")
df_none = df[none]
df_none

,id,title,category,company,format,level,salary,link
6452,28919,разработчик 1c,development,topgo.group,гибрид Питер,не указан,150000.0,https://hirehi.ru/development/razrabotchik-1c-...


Нужно присвоить грейд двум вакансиям без грейда, для этого посмотрим на медианные значения зарплат в их категории. 

In [24]:
df["category"].value_counts()

category
development    3612
analytics      1325
management     1116
qa              914
design          753
devops          596
marketing       438
Name: count, dtype: int64

In [25]:
categories = df.groupby(["category","level"])["salary"].agg(["median"])
categories

median
category    level               
analytics   head        462000.0
            intern       65000.0
            junior      101650.0
            lead        360500.0
            middle      208200.0
            senior      282400.0
design      head        518900.0
            intern       50700.0
            junior       70000.0
            lead        314800.0
            middle      139000.0
            senior      274550.0
            грейда нет  240000.0
development head        606150.0
            intern       60000.0
            junior       80000.0
            lead        412200.0
            middle      224750.0
            senior      336000.0
            не указан   150000.0
devops      intern       53400.0
            junior       97000.0
            lead        411400.0
            middle      231150.0
            senior      343400.0
management  head        496850.0
            intern       67250.0
            junior       98300.0
            lead        388500.0
            middle      224450.0
            senior      339000.0
marketing   head        457100.0
            intern       58900.0
            junior       62350.0
            lead        328000.0
            middle      154900.0
            senior      256300.0
qa          head        519900.0
            intern       50700.0
            junior       81000.0
            lead        342200.0
            middle      194300.0
            senior      277750.0

In [26]:
withnonegrade = categories.loc[["design","development"]]
withnonegrade

median
category    level               
design      head        518900.0
            intern       50700.0
            junior       70000.0
            lead        314800.0
            middle      139000.0
            senior      274550.0
            грейда нет  240000.0
development head        606150.0
            intern       60000.0
            junior       80000.0
            lead        412200.0
            middle      224750.0
            senior      336000.0
            не указан   150000.0

Видим, что вакансия в дизайне ближе к senior, а в разработке - к middle

In [27]:
df.loc[(df["category"] == "design") & (df["level"] == "грейда нет"), "level"] = "senior"
df.loc[(df["category"] == "development") & (df["level"] == "не указан"), "level"] = "middle"

In [28]:
df["level"].value_counts()

level
middle    4236
senior    3090
lead       768
junior     468
intern     129
head        63
Name: count, dtype: int64

Теперь у нас 6 категорий грейдов, теперь можем заполнить медианными значениями пропуски по зарплате, воспользуемся функцией <a href = "https://dfedorov.spb.ru/pandas/Понимание%20функции%20transform%20в%20Pandas.html">transform</a>  

In [29]:
df["salary"] = df["salary"].fillna(df.groupby(["category","level"])["salary"].transform("median"))
df.isna().sum()

id          0
title       0
category    0
company     0
format      0
level       0
salary      0
link        0
dtype: int64

In [30]:
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240000.0,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350000.0,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372000.0,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223000.0,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210000.0,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,200000.0,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150000.0,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,98300.0,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,242500.0,https://hirehi.ru/analytics/sistemnyi-analitik...


In [34]:
df["salary"] = df["salary"].astype(int)
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,240000,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,350000,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,372000,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,223000,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,210000,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,200000,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,150000,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,98300,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,242500,https://hirehi.ru/analytics/sistemnyi-analitik...


<center><h2>Закончили с level и salary</h2></center>